# LlamaIndex와 AgentCore 메모리 - 법률 문서 분석기 (단기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 법률 문서 분석기를 만드는 방법을 보여줍니다. 단일 법률 분석 세션 내에서의 **단기 메모리** 지속성에 초점을 맞추며, 법률 검토 전반에 걸쳐 계약 조항, 판례 및 규정 준수 문제를 기억할 수 있도록 합니다.

## 아키텍처 개요

![LlamaIndex AgentCore 단기 메모리 아키텍처](LlamaIndex-AgentCore-STM-Arch.png)

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형 메모리                                                               |
| 에이전트 사용 사례  | 법률 문서 분석기                                                                 |
| 에이전트 프레임워크 | LlamaIndex                                                                       |
| LLM 모델            | Anthropic Claude 3.7 Sonnet                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, LlamaIndex 에이전트, 법률 분석 도구                       |
| 예제 난이도         | 초급                                                                             |

학습 내용:
- 법률 문서 분석을 위한 AgentCore 메모리 생성
- 법률 워크플로를 위한 LlamaIndex 네이티브 메모리 통합 사용
- 계약 분석을 위한 법률 전용 도구 구축
- 단일 분석 세션 내에서 법률 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 컨텍스트

이 예제에서는 변호사가 단일 법률 검토 세션 내에서 계약을 분석하고, 법적 이슈를 추적하며, 규정 준수 요구사항을 관리하는 "법률 문서 분석기"를 만듭니다. 이 분석기는 AgentCore 메모리를 사용하여 분석 전반에 걸쳐 계약 조항, 위험 평가, 판례 및 규정 준수 문제에 대한 컨텍스트를 유지합니다.

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 의존성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore 메모리 구성

법률 분석기를 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# AgentCore 메모리 리소스 생성
region = os.getenv('AWS_REGION', 'us-east-1')
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f'LegalAnalyzerShortTerm_{int(datetime.now().timestamp())}',
        description='Legal document analyzer short-term memory for single session context',
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )
    memory_id = response['id']
    print(f"AgentCore 메모리 생성 완료: {memory_id}")
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 법률 분석 도구 구현

법률 문서 분석을 위한 전문 도구를 정의합니다:

In [ ]:
def analyze_contract_clause(clause_text: str, clause_type: str, risk_level: str) -> str:
    """Analyze a contract clause and assess its risk level"""
    print(f"{clause_type} 조항 분석 완료 (위험도: {risk_level})")
    return f"Analyzed {clause_type} clause with {risk_level} risk assessment"

def track_legal_issue(issue: str, priority: str, status: str) -> str:
    """Track legal issue with priority and status"""
    print(f"법적 이슈 추적 중: {issue} (우선순위: {priority}, 상태: {status})")
    return f"Now tracking legal issue: {issue}"

def save_legal_precedent(case_name: str, jurisdiction: str, relevance: str) -> str:
    """Save legal precedent with jurisdiction and relevance"""
    print(f"판례 저장 완료: {case_name} ({jurisdiction})")
    return f"Saved legal precedent: {case_name}"

def flag_compliance_issue(regulation: str, violation_type: str, severity: str) -> str:
    """Flag compliance issue with regulation and severity"""
    print(f"{severity} 수준 규정 준수 문제 플래그: {regulation}")
    return f"Flagged compliance issue: {regulation}"

# 에이전트용 도구 객체 생성
legal_tools = [
    FunctionTool.from_defaults(fn=analyze_contract_clause),
    FunctionTool.from_defaults(fn=track_legal_issue),
    FunctionTool.from_defaults(fn=save_legal_precedent),
    FunctionTool.from_defaults(fn=flag_compliance_issue)
]

## 4단계: LlamaIndex 에이전트 구현

단기 메모리 컨텍스트를 가진 법률 분석기 에이전트를 생성합니다:

In [ ]:
# 단기 메모리 구성 (단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="legal-analyst",
    memory_id=memory_id,
    session_id="legal-analysis-session-today",  # 전체에서 동일한 세션
    namespace="/legal-analysis/"
)

# AgentCore 메모리 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 법률 분석기 에이전트 생성
legal_agent = FunctionAgent(
    tools=legal_tools,
    llm=llm,
    verbose=True
)

print("단기 메모리가 적용된 법률 문서 분석기 준비 완료!")

## 5단계: 단기 메모리 기능 테스트

포괄적인 계약 분석 세션을 통해 법률 분석기의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 사건 설정 및 초기화

In [ ]:
# 상세한 컨텍스트로 법률 분석 세션 초기화
# (Johnson & Associates의 Maria Johnson 변호사로서, TechCorp과 DataSoft Inc 간의 $5M 소프트웨어 라이선스 계약 분석)
response = await legal_agent.run(
    "I'm Attorney Maria Johnson from Johnson & Associates, analyzing a $5M software licensing agreement "
    "between TechCorp (licensor) and DataSoft Inc (licensee). Track this as 'Software License Review' "
    "with critical priority and active status. Contract value: $5M over 3 years.",
    memory=agentcore_memory
)

print("사건 설정:")
print(response)

### 테스트 2: 계약 조항 분석

In [ ]:
# 구체적인 조건이 있는 책임 제한 조항 분석
# (TechCorp의 총 책임은 직접 손해에 대해 $50,000을 초과하지 않으며 모든 간접/결과적/징벌적 손해 배제 - 계약 가치 대비 낮은 상한으로 높은 위험)
response = await legal_agent.run(
    "Analyze this liability clause: 'TechCorp's total liability shall not exceed $50,000 for any direct damages "
    "and excludes all indirect, consequential, or punitive damages.' This is a 'Liability Limitation' clause "
    "with 'High' risk level due to low cap vs contract value.",
    memory=agentcore_memory
)

print("책임 제한 조항 분석:")
print(response)

In [ ]:
# 통지 기간이 있는 해지 조항 분석
# (어느 당사자든 90일 서면 통지로 해지 가능, TechCorp은 중대한 위반 또는 30일 초과 미지급 시 즉시 해지 가능 - 중간 위험)
response = await legal_agent.run(
    "Analyze termination clause: 'Either party may terminate with 90 days written notice. "
    "TechCorp may terminate immediately for material breach or non-payment exceeding 30 days.' "
    "This is a 'Termination' clause with 'Medium' risk level.",
    memory=agentcore_memory
)

print("해지 조항 분석:")
print(response)

### 테스트 3: 계약 컨텍스트 회상

In [ ]:
# 계약 컨텍스트 및 위험 평가 회상 테스트
# (어떤 계약을 분석 중인지, 당사자, 가치, 책임 상한 평가 질문)
response = await legal_agent.run(
    "What contract am I analyzing? Who are the parties, what's the value, and what's my assessment of the liability cap?",
    memory=agentcore_memory
)

print("계약 컨텍스트 회상:")
print(response)
print("\n예상 결과: TechCorp/DataSoft, $5M 계약, $50K 책임 상한 (높은 위험)")

### 테스트 4: 상세 조항 회상

In [ ]:
# 특정 조항 세부 사항 회상 테스트
# (확인된 정확한 해지 통지 기간과 즉시 해지 트리거 질문)
response = await legal_agent.run(
    "What are the exact termination notice periods I found? What triggers immediate termination?",
    memory=agentcore_memory
)

print("해지 세부 사항 회상:")
print(response)
print("\n예상 결과: 90일 통지, 위반 또는 30일 이상 미지급 시 즉시 해지")

### 테스트 5: 판례 통합

In [ ]:
# 사건 세부 정보가 있는 관련 판례 저장
# (TechSoft Inc. v. MegaCorp - 델라웨어 상급법원 - 핵심적 관련성 - 계약 가치 1% 미만의 책임 상한은 소프트웨어 라이선스에서 비양심적이라고 판결)
response = await legal_agent.run(
    "Save legal precedent: 'TechSoft Inc. v. MegaCorp' from 'Delaware Superior Court' with 'Critical' relevance. "
    "This case established that liability caps below 1% of contract value are unconscionable in software licensing.",
    memory=agentcore_memory
)

print("판례 저장 완료:")
print(response)

### 테스트 6: 위험 평가 추론

In [ ]:
# 위험 평가 추론 테스트
# (책임 조항을 높은 위험으로 평가한 이유와 상한과 계약 가치 간의 수학적 관계 질문)
response = await legal_agent.run(
    "Why did I assess the liability clause as high risk? What's the mathematical relationship between the cap and contract value?",
    memory=agentcore_memory
)

print("위험 평가 추론:")
print(response)
print("\n예상 결과: $50K 상한 vs $5M 계약 = 1% 비율, 낮은 비율로 인해 높은 위험")

### 테스트 7: 규정 준수 문제 플래그

In [ ]:
# 규제 세부 사항이 있는 규정 준수 문제 플래그
# (GDPR 제82조 데이터 보호 - 부적절한 책임 보장 - 심각 수준 - $50K 상한은 연간 수익의 4%까지의 잠재적 GDPR 벌금에 불충분)
response = await legal_agent.run(
    "Flag compliance issue: 'GDPR Article 82 Data Protection' violation type 'Inadequate Liability Coverage' "
    "with 'Critical' severity. The $50K cap is insufficient for potential GDPR fines up to 4% of annual revenue.",
    memory=agentcore_memory
)

print("GDPR 규정 준수 문제:")
print(response)

### 테스트 8: 판례 적용

In [ ]:
# 현재 사건에 대한 판례 적용 테스트
# (TechSoft v. MegaCorp 판례가 현재 계약 분석에 어떻게 적용되는지, 책임 조항에 대한 시사점 질문)
response = await legal_agent.run(
    "How does the TechSoft v. MegaCorp precedent apply to my current contract analysis? "
    "What does it suggest about the liability clause?",
    memory=agentcore_memory
)

print("판례 적용:")
print(response)
print("\n예상 결과: 두 사건 모두 ~1% 책임 상한, 판례가 비양심성을 시사")

### 테스트 9: 종합 위험 평가

In [ ]:
# 종합 위험 평가 질의
# (DataSoft에 대한 종합 위험 평가 - 식별된 모든 위험, 심각도 수준 및 관련 판례 요청)
response = await legal_agent.run(
    "Provide a comprehensive risk assessment for DataSoft: What are all the risks I've identified, "
    "their severity levels, and supporting precedents?",
    memory=agentcore_memory
)

print("종합 위험 평가:")
print(response)
print("\n예상 결과: 높은 위험 책임 상한, GDPR 규정 준수 문제, TechSoft 판례 근거")

## 6단계: 세션 경계 테스트

다른 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다:

In [ ]:
# 다른 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="legal-analyst",
    memory_id=memory_id,
    session_id="different-legal-session",  # 다른 세션 ID
    namespace="/legal-analysis/"
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
# (어떤 계약을 분석 중인지, 책임 상한과 규정 준수 문제 질문)
response = await legal_agent.run(
    "What contracts am I analyzing? What liability caps and compliance issues have I found?",
    memory=new_session_memory
)

print("세션 경계 테스트 (다른 세션):")
print(response)
print("\n예상 결과: 이전 세션에서의 회상이 제한적이거나 없음 (단기 메모리 경계)")

In [ ]:
# 원래 세션으로 돌아가서 지속성 확인
# (원래 세션으로 돌아와서 정확한 책임 상한 금액과 GDPR 규정 준수 문제 질문)
response = await legal_agent.run(
    "Back in my original session - what was the exact liability cap amount and GDPR compliance issue I identified?",
    memory=agentcore_memory  # 원래 세션 메모리
)

print("원래 세션 복귀:")
print(response)
print("\n예상 결과: $50K 상한, GDPR 제82조 문제 전체 회상")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀들을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 세션 초반 정보를 회상할 수 있는지 확인"""
        has_content = len(response) > 50
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내에서 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동합니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하며, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 중요한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상
response1 = await legal_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리
response2 = await legal_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 기능
response3 = await legal_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 노트북에서 다음을 시연했습니다:

- **단기 메모리 통합**: 세션 범위 법률 분석을 위해 LlamaIndex와 AgentCore 메모리 사용

- **법률 전용 도구**: 계약 조항 분석, 판례 관리 및 규정 준수 추적

- **문맥적 법률 분석**: 분석기가 계약 세부 사항, 위험 평가 및 판례를 기억

- **위험 평가 추론**: 책임 상한을 계약 가치 및 법적 판례와 연결

- **세션 경계**: 서로 다른 법률 분석 세션 간의 메모리 격리

- **규정 준수 관리**: 규제 문제 및 심각도 수준 추적

법률 문서 분석기는 단기 메모리가 어떻게 단일 세션 내에서 포괄적인 계약 분석을 가능하게 하면서 서로 다른 법률 사안 간에 명확한 경계를 유지하는지 보여줍니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다:

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")